# Crime Forecasting (World Bank Indicators)

This Colab-ready notebook trains a baseline forecasting model for Intentional homicides (per 100k) using World Bank indicators.

What you get:
- Clean country-year panel directly from the World Bank API (no uploads required)
- Feature engineering (lags, rolling means)
- Train/valid/test split by year
- LightGBM baseline with MAE metric
- Forecast utility for 2-year and 5-year horizons per country
- Artifacts export (model + columns)

To run in Colab: Runtime → Change runtime type → Python 3. Then run all cells top-to-bottom.

In [ ]:
!pip -q install lightgbm requests joblib

In [ ]:
import pandas as pd
import numpy as np
import requests
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor
import joblib, json, os

In [ ]:
WB_BASE = 'https://api.worldbank.org/v2'
LABEL = 'SH.STA.HOMIC.ZS'  # Intentional homicides per 100k
FEATURES = [
    'SP.POP.TOTL',
    'NY.GDP.PCAP.CD',
    'SL.UEM.TOTL.ZS',
    'SP.URB.TOTL.IN.ZS',
    'SP.DYN.TFRT.IN',
    'SP.DYN.LE00.IN',
]
def wb_indicator(ind, per_page=20000):
    url = f"{WB_BASE}/country/all/indicator/{ind}?format=json&per_page={per_page}"
    r = requests.get(url).json()
    df = pd.DataFrame(r[1])[["countryiso3code","date","value"]]
    return df.rename(columns={'countryiso3code':'iso3','date':'year','value':ind})

label_df = wb_indicator(LABEL)
feat_dfs = [wb_indicator(c) for c in FEATURES]
df = label_df.copy()
for f in feat_dfs:
    df = df.merge(f, on=['iso3','year'], how='left')
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df = df.dropna(subset=[LABEL]).sort_values(['iso3','year']).reset_index(drop=True)
df.head()

In [ ]:
def add_lags(group, cols, lags=(1,2,3), windows=(3,5)):
    g = group.copy()
    for c in cols:
        for L in lags:
            g[f'{c}_lag{L}'] = g[c].shift(L)
        for w in windows:
            g[f'{c}_roll{w}'] = g[c].rolling(w, min_periods=2).mean()
    return g

cols = [LABEL] + FEATURES
df = df.groupby('iso3', group_keys=False).apply(add_lags, cols=cols)
df_model = df.dropna().reset_index(drop=True)
df_model.shape

In [ ]:
train = df_model[df_model['year'] <= 2016]
valid = df_model[(df_model['year'] > 2016) & (df_model['year'] <= 2019)]
test  = df_model[df_model['year'] > 2019]
target = LABEL
X_cols = [c for c in df_model.columns if c not in ['iso3','year', target]]
len(X_cols), train.shape, valid.shape, test.shape

In [ ]:
model = LGBMRegressor(
    n_estimators=1200, learning_rate=0.02, max_depth=-1,
    num_leaves=31, subsample=0.8, colsample_bytree=0.8, random_state=42
)
model.fit(train[X_cols], train[target],
          eval_set=[(valid[X_cols], valid[target])],
          eval_metric='l1', verbose=100)
pred = model.predict(test[X_cols])
mae = mean_absolute_error(test[target], pred)
print('Test MAE:', mae)

In [ ]:
def forecast_country(df_all, iso3, h=5):
    g = df_all[df_all['iso3']==iso3].sort_values('year').copy()
    if g.empty: return []
    g = add_lags(g, cols, lags=(1,2,3), windows=(3,5))
    g = g.dropna().copy()
    if g.empty: return []
    last_year = int(g['year'].max())
    out = []
    state = g.iloc[-1:].copy()
    for step in range(1, h+1):
        yhat = float(model.predict(state[X_cols])[0])
        new_row = state.iloc[-1:].copy()
        new_row['year'] = last_year + step
        new_row[target] = yhat
        g = pd.concat([g, new_row], ignore_index=True)
        g = add_lags(g, cols, lags=(1,2,3), windows=(3,5))
        state = g.iloc[-1:].copy()
        out.append({'year': last_year + step, 'value': yhat})
    return out

iso3 = 'IND'
print('2-year:', forecast_country(df, iso3, h=2))
print('5-year:', forecast_country(df, iso3, h=5))

In [ ]:
os.makedirs('artifacts', exist_ok=True)
joblib.dump(model, 'artifacts/crime_model_lgbm.pkl')
with open('artifacts/columns.json','w') as f: json.dump({'X_cols': X_cols, 'label': LABEL, 'features': FEATURES}, f)
print('Saved artifacts to artifacts/')